In [33]:
import sys
import os
sys.path.append(os.path.abspath('..'))
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)


You should consider upgrading via the 'C:\Users\luccas.fernandes\Desktop\programacao\consumption-api-gov\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [34]:
import pandas as pd

from data.merges import (
    merge_executor_with_uf,
    merge_meta_with_uf,
    merge_finalidade_with_uf,
    merge_documento_with_uf,
    merge_ordem_pagamento_with_uf,
    merge_historico_pagamento_with_uf,
    merge_plano_trabalho_with_uf,
    merge_relatorio_gestao_with_uf
)
from data.conversions import export_to_excel

In [35]:
from api.endpoints import (
    get_plano_acao_especial, get_executor_especial, get_meta_especial,
    get_finalidade_especial, get_empenho_especial, get_documento_habil_especial,
    get_ordem_pagamento_ordem_bancaria_especial, get_historico_pagamento_especial,
    get_plano_trabalho_especial, get_relatorio_gestao_especial
)
from data.processing import to_dataframe

In [36]:
df_planos = to_dataframe(get_plano_acao_especial())
df_executores = to_dataframe(get_executor_especial())
df_metas = to_dataframe(get_meta_especial())
df_finalidades = to_dataframe(get_finalidade_especial())
df_empenhos = to_dataframe(get_empenho_especial())
df_documentos = to_dataframe(get_documento_habil_especial())
df_ordens = to_dataframe(get_ordem_pagamento_ordem_bancaria_especial())
df_historicos = to_dataframe(get_historico_pagamento_especial())
df_planos_trabalho = to_dataframe(get_plano_trabalho_especial())
df_relatorios = to_dataframe(get_relatorio_gestao_especial())

In [37]:
# --- Filtros ---

# 1. Plano de Ação Especial
df_planos_pe_2020 = df_planos[
    (df_planos['uf_beneficiario_plano_acao'] == 'PE') &
    (df_planos['ano_plano_acao'] >= 2020)
]

print(df_planos.columns)

Index(['id_plano_acao', 'codigo_plano_acao', 'ano_plano_acao',
       'modalidade_plano_acao', 'situacao_plano_acao',
       'cnpj_beneficiario_plano_acao', 'nome_beneficiario_plano_acao',
       'uf_beneficiario_plano_acao', 'codigo_banco_plano_acao',
       'codigo_situacao_dado_bancario_plano_acao', 'nome_banco_plano_acao',
       'numero_agencia_plano_acao', 'dv_agencia_plano_acao',
       'numero_conta_plano_acao', 'dv_conta_plano_acao',
       'nome_parlamentar_emenda_plano_acao',
       'ano_emenda_parlamentar_plano_acao',
       'codigo_parlamentar_emenda_plano_acao',
       'sequencial_emenda_parlamentar_plano_acao',
       'numero_emenda_parlamentar_plano_acao',
       'codigo_emenda_parlamentar_formatado_plano_acao',
       'codigo_descricao_areas_politicas_publicas_plano_acao',
       'descricao_programacao_orcamentaria_plano_acao',
       'motivo_impedimento_plano_acao', 'valor_custeio_plano_acao',
       'valor_investimento_plano_acao', 'id_programa'],
      dtype='object

In [38]:
# 2. Empenho Especial
df_empenhos_pe_2020 = df_empenhos[
    (df_empenhos['uf_beneficiario_empenho'] == 'PE') &
    (pd.to_datetime(df_empenhos['data_emissao_empenho']).dt.year >= 2020)
]

In [39]:
# 3. Executor Especial
df_executores_pe = merge_executor_with_uf(df_executores, df_planos)
print(df_executores_pe.columns)
df_executores_pe_2020 = df_executores_pe[
    (df_executores_pe['uf_beneficiario_plano_acao'] == 'PE') &
    (df_executores_pe['ano_plano_acao'] >= 2020)
]

Index(['id_plano_acao', 'id_executor', 'cnpj_executor', 'nome_executor',
       'objeto_executor', 'vl_custeio_executor', 'vl_investimento_executor',
       'uf_beneficiario_plano_acao', 'ano_plano_acao'],
      dtype='object')


In [40]:
# 4. Meta Especial
df_metas_pe = merge_meta_with_uf(df_metas, df_executores, df_planos)
df_metas_pe_2020 = df_metas_pe[
    (df_metas_pe['uf_beneficiario_plano_acao'] == 'PE') &
    (df_metas_pe['ano_plano_acao'] >= 2020)
]

In [41]:
# 5. Finalidade Especial
df_finalidades_pe = merge_finalidade_with_uf(df_finalidades, df_executores, df_planos)
df_finalidades_pe_2020 = df_finalidades_pe[
    (df_finalidades_pe['uf_beneficiario_plano_acao'] == 'PE') &
    (df_finalidades_pe['ano_plano_acao'] >= 2020)
]

In [42]:
# 6. Documento Hábil Especial
df_documentos_pe = merge_documento_with_uf(df_documentos, df_empenhos, df_planos)
df_documentos_pe_2020 = df_documentos_pe[
    (df_documentos_pe['uf_beneficiario_plano_acao'] == 'PE') &
    (pd.to_datetime(df_documentos_pe['data_emissao_dh']).dt.year >= 2020)
]

In [43]:
# 7. Ordem Pagamento Ordem Bancária Especial
df_ordens_pe = merge_ordem_pagamento_with_uf(df_ordens, df_documentos, df_empenhos, df_planos)
df_ordens_pe_2020 = df_ordens_pe[
    (df_ordens_pe['uf_beneficiario_plano_acao'] == 'PE') &
    (pd.to_datetime(df_ordens_pe['data_emissao_op']).dt.year >= 2020)
]

In [44]:
# 8. Histórico Pagamento Especial
df_historicos_pe = merge_historico_pagamento_with_uf(df_historicos, df_ordens, df_documentos, df_empenhos, df_planos)
df_historicos_pe_2020 = df_historicos_pe[
    (df_historicos_pe['uf_beneficiario_plano_acao'] == 'PE') &
    (pd.to_datetime(df_historicos_pe['data_hora_historico_op']).dt.year >= 2020)
]

In [45]:
# 9. Plano Trabalho Especial
df_planos_trabalho_pe = merge_plano_trabalho_with_uf(df_planos_trabalho, df_planos)
df_planos_trabalho_pe_2020 = df_planos_trabalho_pe[
    (df_planos_trabalho_pe['uf_beneficiario_plano_acao'] == 'PE') &
    (df_planos_trabalho_pe['ano_plano_acao'] >= 2020)
]

In [46]:
# 10. Relatório Gestão Especial
df_relatorios_pe = merge_relatorio_gestao_with_uf(df_relatorios, df_planos)
df_relatorios_pe_2020 = df_relatorios_pe[
    (df_relatorios_pe['uf_beneficiario_plano_acao'] == 'PE') &
    (df_relatorios_pe['ano_plano_acao'] >= 2020)
]

In [47]:
print("Plano de Ação Especial")
display(df_planos_pe_2020.head())

print("Empenho Especial")
display(df_empenhos_pe_2020.head())

print("Executor Especial")
display(df_executores_pe_2020.head())

print("Meta Especial")
display(df_metas_pe_2020.head())

print("Finalidade Especial")
display(df_finalidades_pe_2020.head())

print("Documento Hábil Especial")
display(df_documentos_pe_2020.head())

print("Ordem Pagamento Ordem Bancária Especial")
display(df_ordens_pe_2020.head())

print("Histórico Pagamento Especial")
display(df_historicos_pe_2020.head())

print("Plano Trabalho Especial")
display(df_planos_trabalho_pe_2020.head())

print("Relatório Gestão Especial")
display(df_relatorios_pe_2020.head())

Plano de Ação Especial


,id_plano_acao,codigo_plano_acao,ano_plano_acao,modalidade_plano_acao,situacao_plano_acao,cnpj_beneficiario_plano_acao,nome_beneficiario_plano_acao,uf_beneficiario_plano_acao,codigo_banco_plano_acao,codigo_situacao_dado_bancario_plano_acao,...,codigo_parlamentar_emenda_plano_acao,sequencial_emenda_parlamentar_plano_acao,numero_emenda_parlamentar_plano_acao,codigo_emenda_parlamentar_formatado_plano_acao,codigo_descricao_areas_politicas_publicas_plano_acao,descricao_programacao_orcamentaria_plano_acao,motivo_impedimento_plano_acao,valor_custeio_plano_acao,valor_investimento_plano_acao,id_programa
110,3305,0903-003305,2020,Especial,CIENTE,10091536000113,MUNICIPIO DE CARUARU,PE,104,4.00,...,3080,11,202030800011,202030800011-Daniel Coelho,15-Urbanismo / 451-Infraestrutura Urbana,None,None,0.00,"1,750,000.00",3
167,3362,0903-003362,2020,Especial,CIENTE,10132777000163,MUNICIPIO DE CANHOTINHO,PE,001,4.00,...,3913,4,202039130004,202039130004-André Ferreira,22-Indústria / 661-Promoção Industrial,None,None,0.00,"300,000.00",3
173,3368,0903-003368,2020,Especial,CIENTE,10150068000100,MUNICIPIO DE CONDADO,PE,001,4.00,...,3913,4,202039130004,202039130004-André Ferreira,15-Urbanismo / 451-Infraestrutura Urbana,None,None,0.00,"400,000.00",3
355,3556,0903-003556,2020,Especial,CIENTE,10296887000160,MUNICIPIO DE VERTENTES,PE,001,4.00,...,3913,4,202039130004,202039130004-André Ferreira,15-Urbanismo / 451-Infraestrutura Urbana,None,None,0.00,"300,000.00",3
600,3832,0903-003832,2020,Especial,CIENTE,11049806000190,MUNICIPIO DE CHA GRANDE,PE,001,4.00,...,3913,4,202039130004,202039130004-André Ferreira,15-Urbanismo / 451-Infraestrutura Urbana,1053 - INFRA ESTRUTURA URBANA,None,0.00,"800,000.00",3


Empenho Especial


,id_empenho,id_minuta_empenho,numero_empenho,situacao_empenho,descricao_situacao_empenho,tipo_documento_empenho,descricao_tipo_documento_empenho,status_processamento_empenho,ug_responsavel_empenho,ug_emitente_empenho,...,categoria_despesa_empenho,modalidade_despesa_empenho,cnpj_beneficiario_empenho,nome_beneficiario_empenho,uf_beneficiario_empenho,numero_ro_empenho,data_emissao_empenho,prioridade_desbloqueio_empenho,valor_empenho,id_plano_acao
103,24329,2023NME000019878,2023NE007496,4,Enviado,1,Empenho Original,ENVIADA,170860,170860,...,CUSTEIO,40,11358116000113,MUNICIPIO DE SERTANIA,PE,2023RO007532,2023-07-06,104,"60,000.00",31626
218,23802,2022NME000018457,None,1,Minuta de Empenho,1,Empenho Original,AGUARDANDO_GERACAO_EMPENHO,170860,170860,...,INVESTIMENTO,40,10347466000111,MUNICIPIO DE FLORES,PE,None,2022-12-13,1,"2,000,000.00",22405
315,14720,2021NME000006283,2021NE004954,4,Enviado,1,Empenho Original,ENVIADA,170860,170860,...,INVESTIMENTO,40,10091593000100,MUNICIPIO DE TAQUARITINGA DO NORTE,PE,2021RO004995,2021-12-29,1201847,"100,000.00",13779
333,14719,2021NME000006282,2021NE004953,4,Enviado,1,Empenho Original,ENVIADA,170860,170860,...,INVESTIMENTO,40,10106243000162,MUNICIPIO DE TACARATU,PE,2021RO004994,2021-12-29,1201845,"250,000.00",13778
394,16850,2022NME000010069,2022NE001369,4,Enviado,1,Empenho Original,ENVIADA,170860,170860,...,INVESTIMENTO,40,11358165000156,MUNICIPIO DE CUSTODIA,PE,2022RO001410,2022-05-17,47,"950,000.00",15188


Executor Especial


...

,id_historico_op_ob,data_hora_historico_op,historico_situacao_op,descricao_historico_situacao_op,id_op_ob,id_dh,id_empenho,id_plano_acao,uf_beneficiario_plano_acao,ano_plano_acao


Plano Trabalho Especial


,id_plano_trabalho,situacao_plano_trabalho,ind_orcamento_proprio_plano_trabalho,data_inicio_execucao_plano_trabalho,data_fim_execucao_plano_trabalho,prazo_execucao_meses_plano_trabalho,id_plano_acao,classificacao_orcamentaria_pt,ind_justificativa_prorrogacao_atraso_pt,ind_justificativa_prorrogacao_paralizacao_pt,justificativa_prorrogacao_pt,uf_beneficiario_plano_acao,ano_plano_acao


Relatório Gestão Especial


,id_relatorio_gestao,situacao_relatorio_gestao,parecer_relatorio_gestao,id_plano_acao,uf_beneficiario_plano_acao,ano_plano_acao
653,160,DISPONIBILIZADO,Concluído conforme documentação em anexo.\n,4170,PE,"2,020.00"
858,635,DISPONIBILIZADO,Contratação de Empresa de Engenharia para Exec...,3832,PE,"2,020.00"


In [48]:
export_to_excel(df_planos_pe_2020, '../excel_with_filter/2020/plano_acao_especial_PE_2020.xlsx')
export_to_excel(df_executores_pe_2020, '../excel_with_filter/2020/executor_especial_PE_2020.xlsx')
export_to_excel(df_metas_pe_2020, '../excel_with_filter/2020/meta_especial_PE_2020.xlsx')
export_to_excel(df_finalidades_pe_2020, '../excel_with_filter/2020/finalidade_especial_PE_2020.xlsx')
export_to_excel(df_empenhos_pe_2020, '../excel_with_filter/2020/empenho_especial_PE_2020.xlsx')
export_to_excel(df_documentos_pe_2020, '../excel_with_filter/2020/documento_habil_especial_PE_2020.xlsx')
export_to_excel(df_ordens_pe_2020, '../excel_with_filter/2020/ordem_pagamento_ordem_bancaria_especial_PE_2020.xlsx')
export_to_excel(df_historicos_pe_2020, '../excel_with_filter/2020/historico_pagamento_especial_PE_2020.xlsx')
export_to_excel(df_planos_trabalho_pe_2020, '../excel_with_filter/2020/plano_trabalho_especial_PE_2020.xlsx')
export_to_excel(df_relatorios_pe_2020, '../excel_with_filter/2020/relatorio_gestao_especial_PE_2020.xlsx')